In [ ]:
from IPython.display import Markdown
import torch
import torch.nn.functional as F
import string
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## data

In [ ]:
with open("../data/usnames.txt", "r") as f:
    names = f.readlines()
names = [n.strip() for n in names]

In [ ]:
names[:3]

## tokenize

In [ ]:
tok2idx = {c: i for i, c in enumerate('.' + string.ascii_lowercase)}
idx2tok = {i: c for c, i in tok2idx.items()}

## modeling

In [ ]:
emb_sz = 3
ctx_len = 2
lr = 0.1

In [ ]:
X, Y = [], []

for name in names:
    ctx = ['.'] * ctx_len
    for c in name + '.':
        x = [tok2idx[c] for c in ctx]
        y = tok2idx[c]
        ctx.append(c); ctx.pop(0)
        X.append(x); Y.append(y)

X = torch.tensor(X)
Y = torch.tensor(Y)

In [ ]:
g = torch.Generator().manual_seed(2147483647)

word_embeddings = torch.randn((27, emb_sz), dtype=torch.float32, generator=g)

# 1st hidden layer with 100 neurons
W1 = torch.randn(emb_sz*ctx_len, 100, generator=g)
b1 = torch.randn(100, generator=g)

# output layer
W2 = torch.randn(100, 27, generator=g)
b2 = torch.randn(27, generator=g)

params = [word_embeddings, W1, b1, W2, b2]

print(f"number of parameters = {sum([p.nelement() for p in params])}")

In [ ]:
for p in params: p.requires_grad = True

In [ ]:
for i in range(1000):
    # lets perform updates in minibatch to make the forward and backward pass more effective
    ixs = torch.randint(0, X.shape[0], (32,))
    # forward pass
    X_emb = word_embeddings[X[ixs]]
    h = torch.tanh(X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[ixs])
    
    # backward pass
    for p in params:
        p.grad = None
    loss.backward()
    # update weights
    for p in params:
        p.data += -lr*p.grad
    
print(loss.item())

In [ ]:
# calculating the total loss

X_emb = word_embeddings[X]
h = torch.tanh(X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)
loss

### figuring out a good learning rate

In [ ]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [ ]:
g = torch.Generator().manual_seed(2147483647)

word_embeddings = torch.randn((27, emb_sz), dtype=torch.float32, generator=g)

# 1st hidden layer with 100 neurons
W1 = torch.randn(emb_sz*ctx_len, 100, generator=g)
b1 = torch.randn(100, generator=g)

# output layer
W2 = torch.randn(100, 27, generator=g)
b2 = torch.randn(27, generator=g)

params = [word_embeddings, W1, b1, W2, b2]

print(f"number of parameters = {sum([p.nelement() for p in params])}")

for p in params: p.requires_grad = True

In [ ]:
losses = []
for i in range(1000):
    # lets perform updates in minibatch to make the forward and backward pass more effective
    ixs = torch.randint(0, X.shape[0], (32,))
    # forward pass
    X_emb = word_embeddings[X[ixs]]
    h = torch.tanh(X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[ixs])
    
    # backward pass
    for p in params:
        p.grad = None
    loss.backward()
    # update weights
    for p in params:
        p.data += -lrs[i]*p.grad

    # track
    losses.append(loss.item())
    
print(loss.item())

In [ ]:
plt.plot(lre, losses);

In [ ]:
plt.plot(lrs, losses);

### making data splits

In [ ]:
def build_dataset(names, ctx_len=ctx_len):
    X, Y = [], []

    for name in names:
        ctx = ['.'] * ctx_len
        for c in name + '.':
            x = [tok2idx[c] for c in ctx]
            y = tok2idx[c]
            ctx.append(c); ctx.pop(0)
            X.append(x); Y.append(y)
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.shuffle(names)
n1 = int(0.8*len(names))
n2 = int(0.9*len(names))

Xtr, Ytr = build_dataset(names[:n1])
Xdev, Ydev = build_dataset(names[n1:n2])
Xte, Yte = build_dataset(names[n2:])

In [ ]:
Xtr.shape, Xdev.shape, Xte.shape

### modeling contd.

In [ ]:
emb_sz = 2
ctx_len = 2
lr = 0.001

In [ ]:
g = torch.Generator().manual_seed(2147483647)

word_embeddings = torch.randn((27, emb_sz), dtype=torch.float32, generator=g)

# 1st hidden layer with 100 neurons
W1 = torch.randn(emb_sz*ctx_len, 300, generator=g)
b1 = torch.randn(300, generator=g)

# output layer
W2 = torch.randn(300, 27, generator=g)
b2 = torch.randn(27, generator=g)

params = [word_embeddings, W1, b1, W2, b2]

print(f"number of parameters = {sum([p.nelement() for p in params])}")

for p in params: p.requires_grad = True

In [ ]:
steps = []
losses = []
for i in range(10000):
    # lets perform updates in minibatch to make the forward and backward pass more effective
    ixs = torch.randint(0, Xtr.shape[0], (32,))
    # forward pass
    X_emb = word_embeddings[Xtr[ixs]]
    h = torch.tanh(X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ixs])
    
    # backward pass
    for p in params:
        p.grad = None
    loss.backward()
    # update weights
    for p in params:
        p.data += -lr*p.grad

    # track
    steps.append(i)
    losses.append(loss.item())
    

    
print(loss.item())

In [ ]:
plt.plot(steps, losses);

In [ ]:
# calculating the loss on full training set

X_emb = word_embeddings[Xtr]
h = torch.tanh(X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)
loss

In [ ]:
# calculating the loss on validation set

X_emb = word_embeddings[Xdev]
h = torch.tanh(X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev)
loss

### visualizing the embeddings

In [ ]:
word_embeddings.shape

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(word_embeddings[:, 0].data, word_embeddings[:, 1].data, s=200)
for i in range(word_embeddings.shape[0]):
    plt.text(word_embeddings[i, 0].item(), word_embeddings[i, 1].item(), idx2tok[i], ha="center", va="center", color="white")
plt.grid("minor")

### scaling up embedding size

In [ ]:
emb_sz = 10
ctx_len = 3

In [ ]:
Xtr, Ytr = build_dataset(names[:n1], ctx_len=ctx_len)
Xdev, Ydev = build_dataset(names[n1:n2], ctx_len=ctx_len)
Xte, Yte = build_dataset(names[n2:], ctx_len=ctx_len)

In [ ]:
g = torch.Generator().manual_seed(2147483647)

word_embeddings = torch.randn((27, emb_sz), dtype=torch.float32, generator=g)

# 1st hidden layer with 100 neurons
W1 = torch.randn(emb_sz*ctx_len, 200, generator=g)
b1 = torch.randn(200, generator=g)

# output layer
W2 = torch.randn(200, 27, generator=g)
b2 = torch.randn(27, generator=g)

params = [word_embeddings, W1, b1, W2, b2]

print(f"number of parameters = {sum([p.nelement() for p in params])}")

for p in params: p.requires_grad = True

In [ ]:
steps = []
losses = []
lr = 0.1
for i in range(200_000):
    # lets perform updates in minibatch to make the forward and backward pass more effective
    ixs = torch.randint(0, Xtr.shape[0], (32,))
    # forward pass
    X_emb = word_embeddings[Xtr[ixs]]
    h = torch.tanh(X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ixs])
    
    # backward pass
    for p in params:
        p.grad = None
    loss.backward()
    # update weights
    if i > 100_000: lr = 0.01
    if i > 150_000: lr = 0.001
    for p in params:
        p.data += -lr*p.grad

    # track
    steps.append(i)
    losses.append(loss.log10().item())
    

    
print(loss.item())

In [ ]:
plt.plot(steps, losses);

In [ ]:
# calculating the loss on full training set

X_emb = word_embeddings[Xtr]
h = torch.tanh(X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)
loss

In [ ]:
# calculating the loss on validation set

X_emb = word_embeddings[Xdev]
h = torch.tanh(X_emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev)
loss

### sampling from the model

In [ ]:
g = torch.Generator().manual_seed(2147483647)
for i in range(25):
    out = []
    ctx = ['.']*ctx_len
    while True:
        emb = word_embeddings[torch.tensor([tok2idx[t] for t in ctx])]
        h = torch.tanh(emb.view(-1, emb_sz*ctx_len) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
        tok = idx2tok[ix]
        ctx.append(tok); ctx.pop(0)
        out.append(tok)
        if tok == '.': break
    print(''.join(out))